In [1]:
import json 
import numpy as np
import os
from ortools.constraint_solver import routing_enums_pb2, pywrapcp
import pandas as pd
import xgboost as xgb

ModuleNotFoundError: No module named 'ortools'

In [ ]:
NUM_VEHICLES = 3
CAP = 100
DEPOT = 0
SCALE = 100
TL = 30

In [ ]:
ZONES = ["Whitefield","Koramangala","Indiranagar","Hebbal","Marathahalli",
         "Electronic City","Jayanagar","Rajajinagar","Yeshwanthpur","BTM Layout",
         "HSR Layout","Bannerghatta Rd","Yelahanka","Sarjapur Road","MG Road","Banashankari"]

ZTYPE = {"Whitefield":"IT","Koramangala":"Commercial","Indiranagar":"Commercial",
         "Hebbal":"Residential","Marathahalli":"IT","Electronic City":"IT",
         "Jayanagar":"Residential","Rajajinagar":"Residential","Yeshwanthpur":"Industrial",
         "BTM Layout":"Residential","HSR Layout":"Commercial","Bannerghatta Rd":"Residential",
         "Yelahanka":"Residential","Sarjapur Road":"IT","MG Road":"Commercial",
         "Banashankari":"Residential"}
         
FEATURES = ["source_id","dest_id","hod","hod_sin","hod_cos","is_peak_morning",
            "is_peak_evening","is_off_peak","is_monsoon","quarter",
            "distance_km","speed_proxy","src_it","src_commercial","src_residential",
            "src_industrial","dst_it","dst_commercial","dst_residential","dst_industrial"]

TARGET = "mean_travel_time_min"

XGB_PARAMS = {"n_estimators":500, "max_depth":6, "learning_rate":0.05,
              "subsample":0.8, "colsample_bytree":0.8, "min_child_weight":5,
              "reg_alpha":0.1, "reg_lambda":1.0, "tree_method":"hist", "random_state":42
}

In [ ]:
# Generate Q10/Q50/Q90 matrices for a given hour using trained models.
def get_matrices_for_hour(hod, dist_df):
    models = {}
    for qi in [10, 50, 90]:
        m = xgb.XGBRegressor()
        m.load_model(os.path.join("models", f"model_q{qi}.json"))
        models[qi] = m
        
    n = len(ZONES)
    rows = []
    for i in range(n):
        for j in range(n):
            if i == j: continue
            d  = dist_df.iloc[i,j]
            st = ZTYPE[ZONES[i]]; dt = ZTYPE[ZONES[j]]
            rows.append({
                "source_id":i,"dest_id":j,"hod":hod,
                "hod_sin":np.sin(2*np.pi*hod/24),"hod_cos":np.cos(2*np.pi*hod/24),
                "is_peak_morning":int(hod in[7,8,9,10]),
                "is_peak_evening":int(hod in[17,18,19,20]),
                "is_off_peak":int(hod not in[7,8,9,10,17,18,19,20]),
                "is_monsoon":1,"quarter":2,
                "distance_km":d,"speed_proxy":d/0.55,
                "src_it":int(st=="IT"),"src_commercial":int(st=="Commercial"),
                "src_residential":int(st=="Residential"),"src_industrial":int(st=="Industrial"),
                "dst_it":int(dt=="IT"),"dst_commercial":int(dt=="Commercial"),
                "dst_residential":int(dt=="Residential"),"dst_industrial":int(dt=="Industrial")
            })
            
    X = pd.DataFrame(rows)[FEATURES]
    preds = {}
    for qi, m in models.items(): 
        preds[qi] = np.maximum(m.predict(X), 1.0)
        
    q10=np.zeros((n,n)); q50=np.zeros((n,n)); q90=np.zeros((n,n))
    idx=0
    for i in range(n):
        for j in range(n):
            if i==j: continue
            q10[i][j]=round(preds[10][idx],2)
            q50[i][j]=round(preds[50][idx],2)
            q90[i][j]=round(preds[90][idx],2)
            idx+=1
    return q10.tolist(), q50.tolist(), q90.tolist()

In [ ]:
# CVRP solve for analysis.
def solve_cvrp(cm, n_customers=15, seed=0):
    np.random.seed(seed)
    n   = n_customers + 1
    dem = [0] + [int(np.random.randint(10,31)) for _ in range(n_customers)]
    ci  = [[int(cm[i][j]*SCALE) for j in range(n)] for i in range(n)]
    mgr = pywrapcp.RoutingIndexManager(n, NUM_VEHICLES, DEPOT)
    rt  = pywrapcp.RoutingModel(mgr)
    def cc(f,t): return ci[mgr.IndexToNode(f)][mgr.IndexToNode(t)]
    def dc(f):   return dem[mgr.IndexToNode(f)]
    ci_ = rt.RegisterTransitCallback(cc)
    di_ = rt.RegisterUnaryTransitCallback(dc)
    rt.SetArcCostEvaluatorOfAllVehicles(ci_)
    rt.AddDimensionWithVehicleCapacity(di_, 0, [CAP]*NUM_VEHICLES, True, "Cap")
    for nd in range(1,n): rt.AddDisjunction([mgr.NodeToIndex(nd)], 100000)
    sp = pywrapcp.DefaultRoutingSearchParameters()
    sp.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    sp.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    sp.time_limit.seconds = TL; sp.log_search = False
    sol = rt.SolveWithParameters(sp)
    if not sol: return 9999
    total = 0
    
    for v in range(NUM_VEHICLES):
        idx = rt.Start(v)
        while not rt.IsEnd(idx):
            ni = sol.Value(rt.NextVar(idx))
            total += ci[mgr.IndexToNode(idx)][mgr.IndexToNode(ni)]; idx=ni
    return round(total/SCALE, 2)

In [ ]:
def analyse():
    print("COST-UNCERTAINTY TRADEOFF ANALYSIS")
    dist_df = pd.read_csv("distance_matrix_osrm.csv", index_col=0)
    # Test key hours: 2am, 7am, 9am, 12pm, 6pm, 10pm
    test_hours = [2, 7, 9, 12, 18, 22]
    labels     = ["Late night","Pre-peak morning","Peak morning","Midday","Peak evening","Night"]
    results    = []
    for hod, label in zip(test_hours, labels):
        q10, q50, q90 = get_matrices_for_hour(hod, dist_df)
        # Solve 5 instances per hour and average
        c10 = np.mean([solve_cvrp(q10,15,s) for s in range(5)])
        c50 = np.mean([solve_cvrp(q50,15,s) for s in range(5)])
        c90 = np.mean([solve_cvrp(q90,15,s) for s in range(5)])
        gap = (c90-c10)/c10*100
        print(f"  {label} (hod={hod}): Q10={c10:.1f} Q50={c50:.1f} Q90={c90:.1f} gap={gap:+.1f}%")
        results.append({
            "time_slot":label, "hod":hod,
            "q10_cost":round(c10,1), "q50_cost":round(c50,1),
            "q90_cost":round(c90,1), "q90_vs_q10_pct":round(gap,1)
        })
    df_results = pd.DataFrame(results)
    df_results.to_csv("cost_tradeoff_analysis.csv", index=False)
    print(f"  Saved: {"cost_tradeoff_analysis.csv"}")
    return df_results

In [ ]:
analyse()